# Projeto 1 - Academia

### Primeiro, vamos importar as bibliotecas necessárias.

In [1]:
import glfw
from OpenGL.GL import *
import OpenGL.GL.shaders
import numpy as np
import glm
import math

### Inicializando janela

In [2]:
glfw.init()
glfw.window_hint(glfw.VISIBLE, glfw.FALSE);
window = glfw.create_window(700, 700, "Academia", None, None)

if (window == None):
    print("Failed to create GLFW window")
    glfwTerminate()
    
glfw.make_context_current(window)

### Shaders

Note que agora usamos vec3, já que estamos em 3D.

In [3]:
vertex_code = """
        attribute vec3 position;
        uniform mat4 mat_transformation;
        void main(){
            gl_Position = mat_transformation * vec4(position,1.0);
        }
        """

In [4]:
fragment_code = """
        uniform vec4 color;
        void main(){
            gl_FragColor = color;
        }
        """

### Requisitando slot para a GPU para nossos programas Vertex e Fragment Shaders

In [5]:
# Request a program and shader slots from GPU
program  = glCreateProgram()
vertex   = glCreateShader(GL_VERTEX_SHADER)
fragment = glCreateShader(GL_FRAGMENT_SHADER)


### Associando nosso código-fonte aos slots solicitados

In [6]:
# Set shaders source
glShaderSource(vertex, vertex_code)
glShaderSource(fragment, fragment_code)

### Compilando o Vertex Shader

Se há algum erro em nosso programa Vertex Shader, nosso app para por aqui.

In [7]:
# Compile shaders
glCompileShader(vertex)
if not glGetShaderiv(vertex, GL_COMPILE_STATUS):
    error = glGetShaderInfoLog(vertex).decode()
    print(error)
    raise RuntimeError("Erro de compilacao do Vertex Shader")


### Compilando o Fragment Shader

Se há algum erro em nosso programa Fragment Shader, nosso app para por aqui.

In [8]:
glCompileShader(fragment)
if not glGetShaderiv(fragment, GL_COMPILE_STATUS):
    error = glGetShaderInfoLog(fragment).decode()
    print(error)
    raise RuntimeError("Erro de compilacao do Fragment Shader")

### Associando os programas compilado ao programa principal

In [9]:
# Attach shader objects to the program
glAttachShader(program, vertex)
glAttachShader(program, fragment)


### Linkagem do programa

In [10]:
# Build program
glLinkProgram(program)
if not glGetProgramiv(program, GL_LINK_STATUS):
    print(glGetProgramInfoLog(program))
    raise RuntimeError('Linking error')
    
# Make program the default program
glUseProgram(program)

### Preparando dados para enviar a GPU

Até aqui, compilamos nossos Shaders para que a GPU possa processá-los.

Por outro lado, as informações de vértices geralmente estão na CPU e devem ser transmitidas para a GPU.


In [11]:
#Criando um cômodo para gerar perspectiva
vertices_comodo = [

    ### CÔMODO ###
    #Linha 1 - horizontal
    (0.0, 0.0, 0.0),
    (1.0, -0.3, -0.0),

    #Linha 2 - horizontal
    (0.0, 0.0, 0.0),
    (-1.0, -0.5, 0.0),

    #Linha 3 - vertical
    (0.0, 0.0,0.0),
    (0.0, 1.0, 0.0)
]

## Importando os objetos desenhados no Blender

In [12]:
def carregar_obj(arq):
    ''' Função recebe um obj que descreve um objeto composto da cena, retorna um dicionário separando os objetos componentes '''
    vertices = []
    objs = {}
    objeto_atual = None

    with open(arq, "r", encoding="utf-8") as f:
        for linha in f:
            linha = linha.strip()

            if not linha or linha.startswith("#"): #ignorando comentários do arquivo obj
                continue

            if linha.startswith("o "): #Salvando os nomes dos objetos componentes
                objeto_atual = linha[2:].strip()
                objs[objeto_atual] = [] #adionando uma entrada no dicionário para o objeto atual
            
            elif linha.startswith("v "): #salvando as coordenadas de cada vértice
                partes = linha.split()
                x = float(partes[1])
                y = float(partes[2])
                z = float(partes[3])
                vertices.append((x, y, z))

            elif linha.startswith("f "): #identificando quais vértices formam cada triângulo
                partes = linha.split()[1:] #ignorando f

                #formato dos dados: 2/1 ou 2/1/3
                indices = []
                for p in partes:
                    indice_vertice = int(p.split("/")[0]) - 1  #obj começa em 1 - pegando somente o primeiro número (índice do vértice)
                    indices.append(indice_vertice)

                #obj já está triangulado, então cada face tem 3 vértices
                for idx in indices:
                    objs[objeto_atual].append(vertices[idx]) #divide em vértices usando os índices lidos

    return objs

In [13]:
objetos = carregar_obj("banco.obj")

print(objetos.keys())

dict_keys(['assento', 'encosto', 'pe2', 'pe1', 'pe4', 'pe3'])


In [14]:
#Juntando todos os vértices
dados_vertices = []

#Adicionando cômodo 
dados_vertices.extend(vertices_comodo)

inicio = len(dados_vertices)

intervalos = {}

for nome_obj, lista_vertices in objetos.items():
    inicio = len(dados_vertices)
    dados_vertices.extend(lista_vertices)
    quantidade = len(lista_vertices)

    intervalos[nome_obj] = (inicio, quantidade)

print(dados_vertices)

[(0.0, 0.0, 0.0), (1.0, -0.3, -0.0), (0.0, 0.0, 0.0), (-1.0, -0.5, 0.0), (0.0, 0.0, 0.0), (0.0, 1.0, 0.0), (-0.035627, 0.008, 0.027214), (-0.035627, -0.008, -0.087625), (-0.035627, -0.008, 0.027214), (-0.035627, 0.008, -0.087625), (0.078373, -0.008, -0.087625), (-0.035627, -0.008, -0.087625), (0.078373, 0.008, -0.087625), (0.078373, -0.008, 0.027214), (0.078373, -0.008, -0.087625), (0.078373, 0.008, 0.027214), (-0.035627, -0.008, 0.027214), (0.078373, -0.008, 0.027214), (0.078373, -0.008, -0.087625), (-0.035627, -0.008, 0.027214), (-0.035627, -0.008, -0.087625), (-0.035627, 0.008, -0.087625), (0.078373, 0.008, 0.027214), (0.078373, 0.008, -0.087625), (-0.035627, 0.008, 0.027214), (-0.035627, 0.008, -0.087625), (-0.035627, -0.008, -0.087625), (-0.035627, 0.008, -0.087625), (0.078373, 0.008, -0.087625), (0.078373, -0.008, -0.087625), (0.078373, 0.008, -0.087625), (0.078373, 0.008, 0.027214), (0.078373, -0.008, 0.027214), (0.078373, 0.008, 0.027214), (-0.035627, 0.008, 0.027214), (-0.0356

## Array final de vértices

Após a leitura de todos os arquivos .obj, agora criamos o array final vertices

In [15]:
vertices = np.zeros(len(dados_vertices), [("position", np.float32, 3)])
vertices["position"] = dados_vertices

### Para enviar nossos dados da CPU para a GPU, precisamos requisitar um slot (buffer).

In [16]:
# Request a buffer slot from GPU
buffer_VBO = glGenBuffers(1)
# Make this buffer the default one
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO)


### Abaixo, nós enviamos todo o conteúdo da variável vertices.

Veja os parâmetros da função glBufferData [https://www.khronos.org/registry/OpenGL-Refpages/gl4/html/glBufferData.xhtml]

In [17]:
# Upload data
glBufferData(GL_ARRAY_BUFFER, vertices.nbytes, vertices, GL_DYNAMIC_DRAW)
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO)

### Associando variáveis do programa GLSL (Vertex Shader) com nossos dados

Primeiro, definimos o byte inicial e o offset dos dados.

In [18]:
# Bind the position attribute
# --------------------------------------
stride = vertices.strides[0]
offset = ctypes.c_void_p(0)


Em seguida, soliciamos à GPU a localização da variável "position" (que guarda coordenadas dos nossos vértices). Nós definimos essa variável no Vertex Shader.

In [19]:
loc = glGetAttribLocation(program, "position")
glEnableVertexAttribArray(loc)

A partir da localização anterior, nós indicamos à GPU onde está o conteúdo (via posições stride/offset) para a variável position (aqui identificada na posição loc).

Outros parâmetros:

* Definimos que possui <b> três </b> coordenadas
* Que cada coordenada é do tipo float (GL_FLOAT)
* Que não se deve normalizar a coordenada (False)

Mais detalhes: https://www.khronos.org/registry/OpenGL-Refpages/gl4/html/glVertexAttribPointer.xhtml

In [20]:
glVertexAttribPointer(loc, 3, GL_FLOAT, False, stride, offset)

### Vamos pegar a localização da variável color para que possamos definir a cor em nosso laço da janela!

In [21]:
loc_color = glGetUniformLocation(program, "color")

### Capturando eventos de teclado e modificando variáveis para a matriz de transformação

In [22]:
def key_event(window,key,scancode,action,mods):
    global angulo_y

    #Rotação do banco
    if key == 65: #rotacionando para a esquerda
        angulo_y += 0.01
    if key == 83: #rotacionando para a direita
        angulo_y -= 0.01

    if key == 82: #reset r
        angulo_y=0.1

glfw.set_key_callback(window,key_event)

### Nesse momento, nós exibimos a janela!


In [23]:
glfw.show_window(window)

# Programa principal

## Funções

In [24]:
def desenhar_triangulos3D(inicio, fim, R, G, B):
    for i in range(inicio, fim, 4):
        glUniform4f(loc_color, R, G, B, 1.0)
        glDrawArrays(GL_TRIANGLE_STRIP, i, 4)

In [25]:
def calcula_transform(angulo_x, angulo_y, angulo_z, tx, ty, tz, sx, sy, sz):
    cos_x = math.cos(angulo_x)
    sin_x = math.sin(angulo_x)
    cos_y = math.cos(angulo_y)
    sin_y = math.sin(angulo_y)
    cos_z = math.cos(angulo_z)
    sin_z = math.sin(angulo_z)
    
    mat_rotation_z = np.array([     cos_z, -sin_z, 0.0, 0.0, 
                                    sin_z,  cos_z, 0.0, 0.0, 
                                    0.0,      0.0, 1.0, 0.0, 
                                    0.0,      0.0, 0.0, 1.0], np.float32)
    
    mat_rotation_x = np.array([     1.0,   0.0,    0.0, 0.0, 
                                    0.0, cos_x, -sin_x, 0.0, 
                                    0.0, sin_x,  cos_x, 0.0, 
                                    0.0,   0.0,    0.0, 1.0], np.float32)
    
    mat_rotation_y = np.array([     cos_y,  0.0, sin_y, 0.0, 
                                    0.0,    1.0,   0.0, 0.0, 
                                    -sin_y, 0.0, cos_y, 0.0, 
                                    0.0,    0.0,   0.0, 1.0], np.float32)
    mat_scale = np.array([sx, 0.0, 0.0, 0.0,
                          0.0, sy, 0.0, 0.0,
                          0.0, 0.0, sz, 0.0,
                          0.0, 0.0, 0.0, 1.0], np.float32)
    
    
    mat_translacao = np.array([     1.0,  0.0, 0.0,     tx, 
                                    0.0,    1.0,   0.0, ty, 
                                    0.0,    0.0,   1.0, tz, 
                                    0.0,    0.0,   0.0, 1.0], np.float32)


    # sequencia de transformações: rotação z, rotação y, rotação x, escala, translação
    mat_transform = multiplica_matriz(mat_rotation_z,mat_rotation_y)
    mat_transform = multiplica_matriz(mat_rotation_x,mat_transform)
    mat_transform = multiplica_matriz(mat_scale,mat_transform)
    mat_transform = multiplica_matriz(mat_translacao,mat_transform) #translacao ultima

    return mat_transform

### Loop principal da janela.

In [26]:
angulo_x = 0.05
angulo_y = 0.1
angulo_z = 0.0

glEnable(GL_DEPTH_TEST) ### importante para 3D

def multiplica_matriz(a,b):
    m_a = a.reshape(4,4)
    m_b = b.reshape(4,4)
    m_c = np.dot(m_a,m_b)
    c = m_c.reshape(1,16)
    return c

loc_transformation = glGetUniformLocation(program, "mat_transformation")

while not glfw.window_should_close(window):
 
    #glPolygonMode(GL_FRONT_AND_BACK,GL_LINE) ----- MUDAR ISSO PRA ESTAR DE ACORDO COM O ENUNCIADO DO PROJETO
        
    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)    
    glClearColor(1.0, 1.0, 1.0, 1.0)

    mat_transform_comodo = np.array([1.0,  0.0, 0.0,     0.0, 
                              0.0,    1.0,   0.0, 0.0,
                              0.0,    0.0,   1.0, 0.0, 
                              0.0,    0.0,   0.0, 1.0], np.float32)
    
    
    glUniformMatrix4fv(loc_transformation, 1, GL_TRUE, mat_transform_comodo) 
    
    glUniform4f(loc_color, 0.0, 0.0, 0.0, 1.0)
    glDrawArrays(GL_LINES, 0, 6)

    #Desenhando cada face do assento
    mat_transform_banco = calcula_transform(angulo_x, angulo_y, angulo_z, -0.05, -0.5, 0, 2.5, 2.5, 2.5)

    glUniformMatrix4fv(loc_transformation, 1, GL_TRUE, mat_transform_banco)
    
    #Adicionando cor no assento + encosto do banco (preto)
    for nome in ["assento", "encosto"]:
        inicio, qtd = intervalos[nome]
        
        glUniform4f(loc_color, 0, 0, 0, 1)
        glDrawArrays(GL_TRIANGLES, inicio, qtd)
        
    for nome in ["pe1", "pe2", "pe3", "pe4"]:
        inicio, qtd = intervalos[nome]
        glUniform4f(loc_color, 0.5, 0.5, 0.5, 1)
        glDrawArrays(GL_TRIANGLES, inicio, qtd)
    
    glfw.swap_buffers(window)
    glfw.poll_events()

glfw.terminate()